In [11]:
#ô code 1 
import pandas as pd


df = pd.read_csv('data/Dataset-Unicauca-Version2-87Atts.csv')
df = df.sample(300000, random_state=42).reset_index(drop=True) #lấy 100k dòng chạy thử và đánh lại số tt chạy từ đầu 
# Lệnh này sẽ gom nhóm và đếm tất cả các loại app có trong file
print(df['ProtocolName'].unique())
print(f"Tổng số ứng dụng khác nhau là: {df['ProtocolName'].nunique()}")

<StringArray>
[             'HTTP',            'GOOGLE',        'HTTP_PROXY',
      'HTTP_CONNECT',         'MICROSOFT',            'AMAZON',
     'CONTENT_FLASH',           'YOUTUBE',             'GMAIL',
               'SSL',    'WINDOWS_UPDATE',          'FACEBOOK',
               'MSN',             'SKYPE',           'DROPBOX',
        'OFFICE_365',           'TWITTER',        'CLOUDFLARE',
        'TEAMVIEWER',             'YAHOO',           'NETFLIX',
           'IP_ICMP',               'DNS',           'SPOTIFY',
      'MS_ONE_DRIVE',          'WHATSAPP',             'APPLE',
         'UBUNTUONE',      'APPLE_ITUNES',       'GOOGLE_MAPS',
              'EBAY',       'SSL_NO_CERT',         'INSTAGRAM',
              'MQTT',            'TIMMEU',          'FTP_DATA',
              'WAZE',              'H323',         'WIKIPEDIA',
      'APPLE_ICLOUD',          'EASYTAXI',            'RADIUS',
           'EDONKEY',     'HTTP_DOWNLOAD',            'DEEZER',
               'TOR',     

In [12]:
#ô code 2
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Mã hóa nhãn: Biến 'YOUTUBE', 'FACEBOOK'... thành số 0, 1, 2...
le = LabelEncoder()
df['ProtocolName_Encoded'] = le.fit_transform(df['ProtocolName'])

# 2. Chọn các cột đặc trưng (Features) - Loại bỏ các cột không phải số hoặc không cần thiết
# Mình tạm loại bỏ các cột IP và ID vì chúng dễ làm AI bị "học vẹt"
features = df.select_dtypes(include=[np.number]).columns.tolist()
features = [f for f in features if f not in ['ProtocolName_Encoded', 'L7Protocol']]

X = df[features]
y = df['ProtocolName_Encoded']

# 3. Xử lý giá trị vô hạn (Inf) hoặc lỗi dữ liệu thường gặp trong lưu lượng mạng
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# 4. Chia dữ liệu: 80% để học, 20% để kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Chuẩn hóa: Đưa các con số về cùng một thang đo (0-1 hoặc tương đương)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Đã chuẩn bị xong dữ liệu cho {len(features)} đặc trưng và {df['ProtocolName'].nunique()} ứng dụng.")


Đã chuẩn bị xong dữ liệu cho 80 đặc trưng và 62 ứng dụng.


In [13]:
#ô code 3 
from sklearn.preprocessing import LabelEncoder

#  Tái mã hóa nhãn riêng cho tập Train và Test để đảm bảo tính liên tục (0, 1, 2...)
# Bước này cực kỳ quan trọng để sửa lỗi "Invalid classes inferred"
le_final = LabelEncoder()
y_train_fixed = le_final.fit_transform(y_train)

# Đối với tập Test, chúng ta chỉ lấy những nhãn mà tập Train đã học được
# Những nhãn nào tập Train không có sẽ bị loại bỏ ở tập Test để không gây lỗi
test_mask = y_test.isin(le_final.classes_)
X_test_fixed = X_test_scaled[test_mask]
y_test_fixed = le_final.transform(y_test[test_mask])

In [14]:
#ô code 4 : XGBOOTS
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Khởi tạo mô hình XGBoost
model_baseline = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Huấn luyện mô hình
print(f"Đang huấn luyện mô hình cơ sở trên {len(le_final.classes_)} ứng dụng...")
model_baseline.fit(X_train_scaled, y_train_fixed)

# 3. Dự đoán và đánh giá
y_pred = model_baseline.predict(X_test_fixed)
print(f"\n--- KẾT QUẢ HOÀN THÀNH WEEK 3 ---")
print(f"Độ chính xác (Accuracy): {accuracy_score(y_test_fixed, y_pred):.4f}")
print("\nBáo cáo chi tiết (Classification Report):")
print(classification_report(y_test_fixed, y_pred))

Đang huấn luyện mô hình cơ sở trên 61 ứng dụng...

--- KẾT QUẢ HOÀN THÀNH WEEK 3 ---
Độ chính xác (Accuracy): 0.7479

Báo cáo chi tiết (Classification Report):
              precision    recall  f1-score   support

           0       0.76      0.52      0.62      1400
           1       0.76      0.39      0.51       139
           2       0.71      0.22      0.33        23
           3       0.50      0.11      0.17        19
           7       0.00      0.00      0.00         1
           8       0.88      0.47      0.61       264
           9       0.99      0.86      0.92       145
          10       0.00      0.00      0.00         1
          11       0.56      0.67      0.61        21
          12       0.92      0.74      0.82       431
          13       1.00      0.14      0.25         7
          14       0.00      0.00      0.00        22
          16       0.89      0.75      0.81       485
          17       0.00      0.00      0.00         1
          18       0.00      

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [15]:
# tinh lọc đặc trưng
importances = model_baseline.feature_importances_
feat_importances = pd.Series(importances, index=X_train.columns)

# Lấy 50 đặc trưng tốt nhất
top_features = feat_importances.nlargest(50).index.tolist()

# Cập nhật lại dữ liệu chỉ dùng 50 cột này
X_train_scaled = X_train_scaled[:, [X_train.columns.get_loc(c) for c in top_features]]
X_test_fixed = X_test_fixed[:, [X_train.columns.get_loc(c) for c in top_features]]

print(f"✅ Đã lọc còn {len(top_features)} đặc trưng tinh túy nhất!")

✅ Đã lọc còn 50 đặc trưng tinh túy nhất!


In [16]:
# ô code 5 : Thiết lập SMOTE
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd
import numpy as np

# 1. Lọc bỏ các class quá ít mẫu như cũ
counts = Counter(y_train_fixed)
valid_classes = [cls for cls, count in counts.items() if count >= 6]
mask = np.isin(y_train_fixed, valid_classes)
X_train_filtered = X_train_scaled[mask]
y_train_filtered = y_train_fixed[mask]

# 2. THIẾT LẬP CHIẾN THUẬT SMOTE THÔNG MINH
current_counts = Counter(y_train_filtered)
# có thể thay con số 2000 này tùy ý (ví dụ 1000, 3000 hoặc 5000)
# Đây là mức "vừa đủ" để AI không bỏ quên app nghèo mà không làm loãng app giàu
target_threshold = 2000 

# Tạo danh sách yêu cầu cho SMOTE
# Nếu app có < 2000 mẫu -> Bơm lên 2000
# Nếu app có > 2000 mẫu -> Giữ nguyên số lượng gốc
sampling_strategy = {
    label: max(count, target_threshold) 
    for label, count in current_counts.items()
}

smote = SMOTE(sampling_strategy=sampling_strategy, k_neighbors=3, random_state=42)

# 3. Chạy SMOTE
print(f"🚀 Đang SMOTE với ngưỡng tối thiểu {target_threshold} mẫu mỗi app...")
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_filtered, y_train_filtered)

# 4. Kiểm tra
new_counts = Counter(y_train_resampled)
print(f"Số lượng mẫu SAU khi SMOTE (đã cân bằng thông minh): {new_counts}")
print(f"Tổng số dòng dữ liệu mới: {len(X_train_resampled)}")

🚀 Đang SMOTE với ngưỡng tối thiểu 2000 mẫu mỗi app...
Số lượng mẫu SAU khi SMOTE (đã cân bằng thông minh): Counter({np.int64(20): 64179, np.int64(23): 46054, np.int64(26): 41886, np.int64(42): 27041, np.int64(24): 21353, np.int64(60): 11449, np.int64(0): 5868, np.int64(29): 3644, np.int64(19): 2661, np.int64(58): 2396, np.int64(39): 2086, np.int64(35): 2000, np.int64(12): 2000, np.int64(50): 2000, np.int64(33): 2000, np.int64(8): 2000, np.int64(16): 2000, np.int64(14): 2000, np.int64(59): 2000, np.int64(55): 2000, np.int64(57): 2000, np.int64(1): 2000, np.int64(31): 2000, np.int64(9): 2000, np.int64(28): 2000, np.int64(27): 2000, np.int64(2): 2000, np.int64(43): 2000, np.int64(21): 2000, np.int64(11): 2000, np.int64(32): 2000, np.int64(25): 2000, np.int64(45): 2000, np.int64(40): 2000, np.int64(13): 2000, np.int64(34): 2000, np.int64(3): 2000, np.int64(48): 2000, np.int64(18): 2000, np.int64(30): 2000, np.int64(51): 2000, np.int64(41): 2000})
Tổng số dòng dữ liệu mới: 290617


In [17]:

import tensorflow as tf
from tensorflow.keras.models import Sequential          
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.layers import BatchNormalization

In [18]:
# ô code 6 
import numpy as np

# dùng X_train_resampled (dữ liệu sau SMOTE)
X_train_cnn = np.expand_dims(X_train_resampled, axis=2)

# X_test thì vẫn giữ nguyên X_test_fixed (vì tập test không được SMOTE)
X_test_cnn = np.expand_dims(X_test_fixed, axis=2)

print(f"✅ Cấu trúc dữ liệu 3D đã sẵn sàng (Đã bao gồm SMOTE): {X_train_cnn.shape}")

✅ Cấu trúc dữ liệu 3D đã sẵn sàng (Đã bao gồm SMOTE): (290617, 50, 1)


In [27]:
#ô code 7
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization

model = Sequential([
    # Thay vì Conv1D, ta dùng LSTM
    # input_shape vẫn là (50, 1) như cũ
    LSTM(128, return_sequences=True, input_shape=(X_train_cnn.shape[1], 1)),
    BatchNormalization(),
    Dropout(0.3),

    LSTM(64), # Lớp LSTM thứ 2 để học sâu hơn
    BatchNormalization(),
    Dropout(0.3),

    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),

    Dense(len(le_final.classes_), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 50, 128)        │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 50, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 61)             │        15,677 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 150,077 (586.24 KB)

 Trainable params: 149,181 (582.74 KB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
# ô code 8
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. Tự động dừng nếu 5 vòng không tăng accuracy (tiết kiệm thời gian cho Mạnh)
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True) 

# 2. Tự động giảm tốc độ học nếu bị "kẹt" (giúp mô hình học sâu hơn)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10)
# Sử dụng y_train_fixed và y_test_fixed để khớp với mã hóa của XGBoost ở trên
history = model.fit(
    X_train_cnn, 
    y_train_resampled, 
    epochs=100, 
    batch_size=32, 
    validation_data=(X_test_cnn, y_test_fixed),
    
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5),
        reduce_lr 
    ]
)

Epoch 1/100
9082/9082 ━━━━━━━━━━━━━━━━━━━━ 582s 64ms/step - accuracy: 0.3650 - loss: 2.2417 - val_accuracy: 0.4820 - val_loss: 1.6745 - learning_rate: 0.0010
Epoch 2/100
9082/9082 ━━━━━━━━━━━━━━━━━━━━ 528s 58ms/step - accuracy: 0.4520 - loss: 1.8783 - val_accuracy: 0.5163 - val_loss: 1.5626 - learning_rate: 0.0010
Epoch 3/100
9082/9082 ━━━━━━━━━━━━━━━━━━━━ 577s 63ms/step - accuracy: 0.4816 - loss: 1.7658 - val_accuracy: 0.5380 - val_loss: 1.4687 - learning_rate: 0.0010
Epoch 4/100
9082/9082 ━━━━━━━━━━━━━━━━━━━━ 655s 72ms/step - accuracy: 0.5005 - loss: 1.6917 - val_accuracy: 0.5588 - val_loss: 1.4166 - learning_rate: 0.0010
Epoch 5/100
9082/9082 ━━━━━━━━━━━━━━━━━━━━ 627s 69ms/step - accuracy: 0.5142 - loss: 1.6378 - val_accuracy: 0.5632 - val_loss: 1.3743 - learning_rate: 0.0010
Epoch 6/100
9082/9082 ━━━━━━━━━━━━━━━━━━━━ 592s 65ms/step - accuracy: 0.5274 - loss: 1.5948 - val_accuracy: 0.5684 - val_loss: 1.3672 - learning_rate: 0.0010
Epoch 7/100
9082/9082 ━━━━━━━━━━━━━━━━━━━━ 637s 70ms

In [ ]:
#ô code 9 
from sklearn.metrics import classification_report
import numpy as np

# Dự đoán trên tập test
y_pred = model.predict(X_test_cnn) #cho mô hình làm bài kiểm tra trên tập test
y_pred_classes = np.argmax(y_pred, axis=1) # chọn ra trong 54 con số xác suất thì ra con số lớn nhất 

## Tìm danh sách các mã số (ID) thực sự xuất hiện trong tập Test
labels_in_test = np.unique(y_test_fixed)

#Lấy tên ứng dụng và ÉP KIỂU SANG STRING 
target_names_in_test = [str(le_final.classes_[i]) for i in labels_in_test]

# In báo cáo chi tiết cho 54 ứng dụng
print(classification_report(y_test_fixed, y_pred_classes, labels=labels_in_test, target_names=target_names_in_test)) # so sánh kết quả kiểm tra với bộ test ban đầu sau đó in ra các nhãn theo tên luôn chứ ko phải alf số 1, 2, 3, ...